# AI Agent Workflow 

**Team: Group1 & Group7**

**Team members:** 
- Ce Chen, 9007166
- Zhuoran Zhang, 9048508
- Haibo Yuan, 9010929
- Abdalla Mohamed,9089339

**GitHub Linke:** https://github.com/chence/AI_Agent_Workshop.git

## Workflow Overview

In this notebook, the complete workflow is organized into ten steps:

1. **Knowledge Base Preparation**  
   Load, clean, and structure the local public-service dataset.

2. **Knowledge Base Expansion**  
   Add new service records and improve catalog coverage.

3. **Data Export and Versioned Catalog**  
   Save the cleaned catalog for reuse and version tracking.

4. **Tool Layer**  
   Build lookup tools for service ownership, routing, and next-step guidance.

5. **Verification Layer**  
   Check whether answers are grounded, usable, and supported by official sources.

6. **Evaluation Dataset**  
   Create ambiguous and harder benchmark questions for testing.

7. **Evaluation Labels**  
   Add labels such as Mixed, Unclear, or expected services for evaluation.

8. **Retrieval Layer**  
   Add retrieval tools such as `retrieve_service_docs` for richer answers.

9. **Evaluation Layer**  
   Compare baseline, retrieval, and tool-using workflows with metrics.

10. **Reflection**  
   Summarize what worked well, limitations, and future improvements.

# 1 Knowledge Base Preparation

In this step, We load the local municipal service catalog.

This catalog will be used as the grounding source for retrieval and tool-based decisions.

In [19]:
import pandas as pd

# Create a small local municipal service catalog
catalog = pd.DataFrame([
    {
        "service_name": "garbage pickup",
        "jurisdiction_level": "Region",
        "responsible_body": "Region of Waterloo Waste Management",
        "description": "Resident garbage collection services.",
        "source_url": "https://www.regionofwaterloo.ca/"
    },
    {
        "service_name": "property tax billing",
        "jurisdiction_level": "City",
        "responsible_body": "City of Kitchener Revenue Division",
        "description": "Property tax billing and payments.",
        "source_url": "https://www.kitchener.ca/"
    },
    {
        "service_name": "snow removal on city streets",
        "jurisdiction_level": "City",
        "responsible_body": "City of Kitchener Operations",
        "description": "Snow clearing for city streets.",
        "source_url": "https://www.kitchener.ca/"
    },
    {
        "service_name": "public health inspections",
        "jurisdiction_level": "Region",
        "responsible_body": "Region of Waterloo Public Health",
        "description": "Restaurant and food safety inspections.",
        "source_url": "https://www.regionofwaterloo.ca/"
    }
])

# Preview dataset
catalog

,service_name,jurisdiction_level,responsible_body,description,source_url
0,garbage pickup,Region,Region of Waterloo Waste Management,Resident garbage collection services.,https://www.regionofwaterloo.ca/
1,property tax billing,City,City of Kitchener Revenue Division,Property tax billing and payments.,https://www.kitchener.ca/
2,snow removal on city streets,City,City of Kitchener Operations,Snow clearing for city streets.,https://www.kitchener.ca/
3,public health inspections,Region,Region of Waterloo Public Health,Restaurant and food safety inspections.,https://www.regionofwaterloo.ca/


# 2 Knowledge Base Expansion 

In [20]:
# Create 3 new service records to expand the catalog
new_services = pd.DataFrame([
    {
        "service_name": "Snow removal",
        "jurisdiction_level": "City",
        "responsible_body": "City of Kitchener",
        "description": "Snow clearing and winter road maintenance within the city.",
        "keywords": "snow; winter; roads; plowing"
    },
    {
        "service_name": "Passport renewal",
        "jurisdiction_level": "Federal",
        "responsible_body": "Government of Canada",
        "description": "Passport application and renewal services for Canadian residents.",
        "keywords": "passport; renewal; travel; federal"
    },
    {
        "service_name": "OHIP card",
        "jurisdiction_level": "Province",
        "responsible_body": "Government of Ontario",
        "description": "Ontario health card services including renewal and updates.",
        "keywords": "ohip; health card; medical; ontario"
    }
])

# Append the new services to the original catalog
catalog_extended = pd.concat([catalog, new_services], ignore_index=True)

# Preview the newly added rows
catalog_extended.tail(3)

,service_name,jurisdiction_level,responsible_body,description,source_url,keywords
4,Snow removal,City,City of Kitchener,Snow clearing and winter road maintenance with...,NaN,snow; winter; roads; plowing
5,Passport renewal,Federal,Government of Canada,Passport application and renewal services for ...,NaN,passport; renewal; travel; federal
6,OHIP card,Province,Government of Ontario,Ontario health card services including renewal...,NaN,ohip; health card; medical; ontario


# 3 Data Export and Versioned Catalog 

In [21]:
# Helper function: normalize the catalog before export
# This keeps text clean and fills missing values.

def normalize_catalog(df):
    """
    Normalize catalog data before exporting.

    Parameters:
        df (pd.DataFrame): Raw or expanded catalog.

    Returns:
        pd.DataFrame: Cleaned catalog.
    """

    clean_df = df.copy()

    # Fill missing values
    clean_df = clean_df.fillna("")

    # Strip spaces from text columns
    for col in clean_df.columns:
        if clean_df[col].dtype == "object":
            clean_df[col] = clean_df[col].astype(str).str.strip()

    return clean_df

In [22]:
# Normalize the updated catalog
clean_catalog_updated = normalize_catalog(catalog_extended)

# Save JSON file
updated_output_path = "service_catalog.cleaned.updated.json"

clean_catalog_updated.to_json(
    updated_output_path,
    orient="records",
    indent=2
)

print("Updated catalog saved.")

clean_catalog_updated.tail()

Updated catalog saved.


,service_name,jurisdiction_level,responsible_body,description,source_url,keywords
2,snow removal on city streets,City,City of Kitchener Operations,Snow clearing for city streets.,https://www.kitchener.ca/,
3,public health inspections,Region,Region of Waterloo Public Health,Restaurant and food safety inspections.,https://www.regionofwaterloo.ca/,
4,Snow removal,City,City of Kitchener,Snow clearing and winter road maintenance with...,,snow; winter; roads; plowing
5,Passport renewal,Federal,Government of Canada,Passport application and renewal services for ...,,passport; renewal; travel; federal
6,OHIP card,Province,Government of Ontario,Ontario health card services including renewal...,,ohip; health card; medical; ontario


In [23]:
# Verify the newly added services in the cleaned catalog

clean_catalog_updated[
    clean_catalog_updated["service_name"].isin([
        "Snow removal",
        "Passport renewal",
        "OHIP card"
    ])
][[
    "service_name",
    "jurisdiction_level",
    "responsible_body",
    "keywords"
]]

,service_name,jurisdiction_level,responsible_body,keywords
4,Snow removal,City,City of Kitchener,snow; winter; roads; plowing
5,Passport renewal,Federal,Government of Canada,passport; renewal; travel; federal
6,OHIP card,Province,Government of Ontario,ohip; health card; medical; ontario


### Result Summary

The cleaned catalog was successfully updated and exported.

Three new services were verified:
- Snow removal
- Passport renewal
- OHIP card

These records can now be used in future retrieval and routing workflows.

# 4  Tool Layer: Service Ownership Lookup

In [37]:
# Tool Layer: Service Ownership Lookup
# This tool looks up service ownership information from the active service catalog.
# It uses the expanded catalog when available.

def lookup_service_tool(service_name: str) -> dict:
    """
    Look up one service in the active catalog by service name.

    Parameters:
        service_name (str): The service name provided by the user or agent.

    Returns:
        dict: A structured result containing service ownership details.
    """

    # Use the expanded catalog if it exists; otherwise use the base catalog
    active_catalog = catalog_extended if "catalog_extended" in globals() else catalog

    # Normalize the user input
    target_name = service_name.strip().lower()

    # Normalize service names before matching
    matches = active_catalog[
        active_catalog["service_name"].astype(str).str.strip().str.lower() == target_name
    ]

    # If no match is found, return a fallback response
    if len(matches) == 0:
        return {
            "service_name": service_name,
            "found": False,
            "jurisdiction_level": "Unknown",
            "responsible_body": "Unknown",
            "description": "No matching service was found in the catalog.",
            "source_url": ""
        }

    # Select the first matching row
    row = matches.iloc[0]

    # Return the structured tool output
    return {
        "service_name": row["service_name"],
        "found": True,
        "jurisdiction_level": row["jurisdiction_level"],
        "responsible_body": row["responsible_body"],
        "description": row["description"],
        "source_url": row["source_url"] if "source_url" in row else ""
    }

In [25]:
# Quick test for the lookup tool

tool_demo = lookup_service_tool("garbage pickup")
tool_demo

{'service_name': 'garbage pickup',
 'found': True,
 'jurisdiction_level': 'Region',
 'responsible_body': 'Region of Waterloo Waste Management',
 'description': 'Resident garbage collection services.',
 'source_url': 'https://www.regionofwaterloo.ca/'}

# 5 Verification Layer: Grounded Answer Checker

In this step, We add a lightweight verification layer that checks whether the answer is grounded in the local catalog and whether it includes a usable official source.

This second-pass check helps the workflow remain more reliable and cautious.

In [26]:
# Verification Layer: Grounded Answer Checker
# This function performs a second-pass check on the tool output.
# It verifies whether the service was found and whether an official source URL is available.

def verify_tool_result(tool_result: dict) -> dict:
    """
    Verify whether the tool result is grounded and usable.

    Parameters:
        tool_result (dict): The structured output returned by the lookup tool.

    Returns:
        dict: A verification summary with groundedness, confidence, and next-step guidance.
    """

    # Check whether the service was successfully found in the catalog
    found = tool_result.get("found", False)

    # Check whether an official source URL exists
    source_url = tool_result.get("source_url", "")
    has_source = isinstance(source_url, str) and len(source_url.strip()) > 0

    # Assign a simple confidence level based on grounded evidence
    if found and has_source:
        confidence = "high"
    elif found:
        confidence = "medium"
    else:
        confidence = "low"

    # Create a practical next step based on the verification result
    if found and has_source:
        next_step = "Use the official source URL to confirm details and contact the responsible body."
    elif found:
        next_step = "The service was found, but the source URL is missing. Verify details on the official municipal website."
    else:
        next_step = "No matching service was found. Try a broader search or check the official municipal website."

    # Overall groundedness decision
    is_grounded = found and has_source

    return {
        "is_grounded": is_grounded,
        "confidence": confidence,
        "has_source": has_source,
        "next_step": next_step
    }

In [27]:
# This helper function combines the tool output with the verification result
# and creates a clean final answer for the user.

def build_verified_answer(tool_result: dict, verification_result: dict) -> str:
    """
    Build a final user-facing answer using the tool result and verification result.

    Parameters:
        tool_result (dict): Structured service lookup output.
        verification_result (dict): Verification summary.

    Returns:
        str: A grounded final answer.
    """

    # If the service was not found, return a cautious fallback answer
    if not tool_result.get("found", False):
        return (
            f"I could not find a matching service for '{tool_result.get('service_name', 'this request')}'.\n\n"
            f"Confidence: {verification_result['confidence']}\n"
            f"Next step: {verification_result['next_step']}"
        )

    # Build a grounded answer when the service exists
    answer = (
        f"Service: {tool_result['service_name']}\n"
        f"Jurisdiction level: {tool_result['jurisdiction_level']}\n"
        f"Responsible body: {tool_result['responsible_body']}\n"
        f"Description: {tool_result['description']}\n"
        f"Source URL: {tool_result['source_url']}\n\n"
        f"Confidence: {verification_result['confidence']}\n"
        f"Next step: {verification_result['next_step']}"
    )

    return answer

In [28]:
# Quick test for the verification layer

verification_demo = verify_tool_result(tool_demo)
verification_demo

{'is_grounded': True,
 'confidence': 'high',
 'has_source': True,
 'next_step': 'Use the official source URL to confirm details and contact the responsible body.'}

In [29]:
# Build the final verified answer

final_verified_answer = build_verified_answer(tool_demo, verification_demo)
print(final_verified_answer)

Service: garbage pickup
Jurisdiction level: Region
Responsible body: Region of Waterloo Waste Management
Description: Resident garbage collection services.
Source URL: https://www.regionofwaterloo.ca/

Confidence: high
Next step: Use the official source URL to confirm details and contact the responsible body.


### Verification Result

The verification layer adds a second-pass check after the lookup tool.

It improves the workflow by checking whether the result is grounded in the local catalog and whether an official source URL is present.

This is useful for public-service routing because the system should not sound overly confident when the evidence is weak or missing.

### Official Source Enforcement

In [30]:
# Exercise 2: Always include at least one official source URL

def final_answer_with_source(service_name):
    """
    Build a final answer that always includes one official source URL.
    """

    result = lookup_service_tool(service_name)

    if result["found"]:

        answer = f"""
Service: {result['service_name']}
Jurisdiction: {result['jurisdiction_level']}
Responsible Body: {result['responsible_body']}
Description: {result['description']}
Official Source: {result['source_url']}
"""
    else:
        answer = "No matching service found."

    return answer


# Quick demo
exercise2_demo = final_answer_with_source("garbage pickup")
print(exercise2_demo)


Service: garbage pickup
Jurisdiction: Region
Responsible Body: Region of Waterloo Waste Management
Description: Resident garbage collection services.
Official Source: https://www.regionofwaterloo.ca/



### Hedging and Clarification Handling

Some public-service questions are ambiguous and cannot be answered with full certainty.

In these cases, the workflow should avoid overconfident routing and instead give a cautious answer with clarification guidance.

In [31]:
# Exercise 3: create a question that requires hedging or clarification

# This question is intentionally ambiguous because "road maintenance"
# may involve either the city or the region depending on the road.
hedging_question = "Who should I contact about road maintenance near my area?"

print("Ambiguous question:")
print(hedging_question)

Ambiguous question:
Who should I contact about road maintenance near my area?


In [32]:
# This helper function builds a cautious answer for ambiguous or mixed cases.

def build_hedged_answer(service_name: str) -> str:
    """
    Build a hedged answer when the service is ambiguous or mixed.

    Parameters:
        service_name (str): The service name to check.

    Returns:
        str: A cautious final answer.
    """

    # Look up the service in the catalog
    result = lookup_service_tool(service_name)

    # If the service is not found, return a fallback answer
    if not result["found"]:
        return (
            f"I could not find a clear match for '{service_name}'.\n\n"
            "Please check the official municipal website or provide more details."
        )

    # If the jurisdiction is unclear or mixed, hedge the answer
    if result["jurisdiction_level"] in ["Mixed", "Unclear"]:
        return (
            f"The request for '{result['service_name']}' may involve more than one authority.\n\n"
            f"Jurisdiction label: {result['jurisdiction_level']}\n"
            f"Possible responsible body: {result['responsible_body']}\n"
            f"Description: {result['description']}\n"
            f"Official source: {result['source_url']}\n\n"
            "This answer should be treated cautiously. "
            "You may need to confirm whether the issue belongs to the City, the Region, or another agency."
        )

    # Otherwise return the normal grounded answer
    verification = verify_tool_result(result)
    return build_verified_answer(result, verification)

In [38]:
# Quick demo for hedging behavior

hedging_demo = build_hedged_answer("Road maintenance")
print(hedging_demo)

The request for 'Road maintenance' may involve more than one authority.

Jurisdiction label: Mixed
Possible responsible body: City / Region of Waterloo
Description: Road issues may belong to the city or the region depending on the road.
Official source: nan

This answer should be treated cautiously. You may need to confirm whether the issue belongs to the City, the Region, or another agency.


In [39]:
# Another ambiguous example

hedging_demo_2 = build_hedged_answer("Housing support")
print(hedging_demo_2)

The request for 'Housing support' may involve more than one authority.

Jurisdiction label: Unclear
Possible responsible body: Various agencies
Description: Housing support programs may depend on the type of service and user situation.
Official source: nan

This answer should be treated cautiously. You may need to confirm whether the issue belongs to the City, the Region, or another agency.


### Hedging Result

This example shows that the workflow should not always give a single confident routing answer.

For ambiguous or mixed cases, a better response is to acknowledge uncertainty, show the possible responsible body, and guide the user toward clarification.

# 6 Evaluation Dataset: Ambiguous User Requests

In [35]:
# Create 2 ambiguous service examples
ambiguous_services = pd.DataFrame([
    {
        "service_name": "Road maintenance",
        "jurisdiction_level": "Mixed",
        "responsible_body": "City / Region of Waterloo",
        "description": "Road issues may belong to the city or the region depending on the road.",
        "keywords": "road; repair; maintenance; street"
    },
    {
        "service_name": "Housing support",
        "jurisdiction_level": "Unclear",
        "responsible_body": "Various agencies",
        "description": "Housing support programs may depend on the type of service and user situation.",
        "keywords": "housing; shelter; support; assistance"
    }
])

# Append the ambiguous services to the extended catalog
catalog_extended = pd.concat([catalog_extended, ambiguous_services], ignore_index=True)

# Preview the ambiguous examples
catalog_extended.tail(2)

,service_name,jurisdiction_level,responsible_body,description,source_url,keywords
7,Road maintenance,Mixed,City / Region of Waterloo,Road issues may belong to the city or the regi...,NaN,road; repair; maintenance; street
8,Housing support,Unclear,Various agencies,Housing support programs may depend on the typ...,NaN,housing; shelter; support; assistance


## 6 Evaluation Dataset: Ambiguous and Hard Benchmark Questions

In this step, We build a stronger benchmark dataset containing ambiguous requests and harder real-world questions.

These examples help evaluate whether the workflow can handle uncertainty, mixed ownership, and more complex user requests.

In [53]:
# Create harder benchmark evaluation questions
# This dataset extends the earlier ambiguous examples.

hard_benchmark_questions = pd.DataFrame([
    {
        "question": "Who handles garbage pickup for apartments?",
        "expected_service": "garbage pickup",
        "difficulty": "medium",
        "type": "specific scenario"
    },
    {
        "question": "My road is icy. Who is responsible for snow clearing?",
        "expected_service": "Snow removal",
        "difficulty": "hard",
        "type": "mixed ownership"
    },
    {
        "question": "How do I renew my passport quickly?",
        "expected_service": "Passport renewal",
        "difficulty": "medium",
        "type": "federal service"
    },
    {
        "question": "Where can I update my OHIP card after moving?",
        "expected_service": "OHIP card",
        "difficulty": "hard",
        "type": "provincial service"
    },
    {
        "question": "I need housing help urgently. Who should I contact?",
        "expected_service": "Housing support",
        "difficulty": "hard",
        "type": "unclear case"
    }
])

hard_benchmark_questions

,question,expected_service,difficulty,type
0,Who handles garbage pickup for apartments?,garbage pickup,medium,specific scenario
1,My road is icy. Who is responsible for snow cl...,Snow removal,hard,mixed ownership
2,How do I renew my passport quickly?,Passport renewal,medium,federal service
3,Where can I update my OHIP card after moving?,OHIP card,hard,provincial service
4,I need housing help urgently. Who should I con...,Housing support,hard,unclear case


### Benchmark Design Notes

This benchmark includes:

- normal requests  
- mixed jurisdiction requests  
- provincial / federal services  
- unclear routing cases  
- more realistic user wording

# 7 Evaluation Labels: Ambiguity and Mixed Ownership

In [36]:
# Check the rows that were labeled as Mixed or Unclear
catalog_extended[
    catalog_extended["jurisdiction_level"].isin(["Mixed", "Unclear"])
][["service_name", "jurisdiction_level", "responsible_body"]]

,service_name,jurisdiction_level,responsible_body
7,Road maintenance,Mixed,City / Region of Waterloo
8,Housing support,Unclear,Various agencies


# 8 Retrieval Layer

### Additional Tool: retrieve_service_docs

In [41]:
# Add a default next_steps_hint column if it does not exist

if "next_steps_hint" not in catalog_extended.columns:
    catalog_extended["next_steps_hint"] = "Please check the official source for the next step."

catalog_extended.head()

,service_name,jurisdiction_level,responsible_body,description,source_url,keywords,next_steps_hint
0,garbage pickup,Region,Region of Waterloo Waste Management,Resident garbage collection services.,https://www.regionofwaterloo.ca/,NaN,Please check the official source for the next ...
1,property tax billing,City,City of Kitchener Revenue Division,Property tax billing and payments.,https://www.kitchener.ca/,NaN,Please check the official source for the next ...
2,snow removal on city streets,City,City of Kitchener Operations,Snow clearing for city streets.,https://www.kitchener.ca/,NaN,Please check the official source for the next ...
3,public health inspections,Region,Region of Waterloo Public Health,Restaurant and food safety inspections.,https://www.regionofwaterloo.ca/,NaN,Please check the official source for the next ...
4,Snow removal,City,City of Kitchener,Snow clearing and winter road maintenance with...,NaN,snow; winter; roads; plowing,Please check the official source for the next ...


In [44]:
def retrieve_service_docs(service_name: str) -> dict:
    """
    Retrieve fuller service documentation from the local catalog.
    """

    active_catalog = catalog_extended if "catalog_extended" in globals() else catalog

    matches = active_catalog[
        active_catalog["service_name"].astype(str).str.strip().str.lower()
        == service_name.strip().lower()
    ]

    if len(matches) == 0:
        return {
            "service_name": service_name,
            "description": "No matching service found in the catalog.",
            "responsible_body": "Unknown",
            "next_steps_hint": "Please check the official municipal website for more details.",
            "source_url": ""
        }

    row = matches.iloc[0]

    return {
        "service_name": row["service_name"],
        "description": row["description"],
        "responsible_body": row["responsible_body"],
        "next_steps_hint": row.get(
            "next_steps_hint",
            "Please check the official source for the next step."
        ),
        "source_url": "" if pd.isna(row.get("source_url")) else row.get("source_url")
    }

# 9 Evaluation Layer: Multi-Method Comparison

In this step, we compare three approaches:

1. Keyword baseline  
2. Retrieval-based workflow  
3. Tool-using agent

We evaluate them on benchmark questions to measure grounding, clarity, routing quality, and practical usefulness.

The goal is to understand how system design improves performance beyond a simple baseline.

In [54]:
# Keyword baseline
# This is a simple non-tool baseline that relies on keyword rules only.

def keyword_baseline(question: str) -> str:
    """
    Generate a simple keyword-based baseline answer.

    Parameters:
        question (str): A user question.

    Returns:
        str: A simple answer based on keyword matching only.
    """

    q = question.lower()

    if "garbage" in q:
        return "Garbage pickup is likely handled by the Region of Waterloo."
    elif "property tax" in q:
        return "Property tax billing is likely handled by the City of Kitchener."
    elif "snow" in q or "icy" in q or "road" in q:
        return "Snow or road-related issues may depend on the city or region."
    elif "passport" in q:
        return "Passport services are handled by the Government of Canada."
    elif "ohip" in q or "health card" in q:
        return "OHIP or health card services are handled by the Government of Ontario."
    elif "housing" in q:
        return "Housing support may involve different agencies depending on the situation."
    else:
        return "The responsible service is unclear. Please check the official website."

In [55]:
# Retrieval-based workflow
# This method uses the retrieval tool to fetch fuller service documentation.

def retrieval_based_answer(service_name: str) -> str:
    """
    Build a final answer using the retrieval layer.

    Parameters:
        service_name (str): The service name to retrieve.

    Returns:
        str: A retrieval-based answer.
    """

    docs = retrieve_service_docs(service_name)

    return (
        f"Service: {docs['service_name']}\n"
        f"Description: {docs['description']}\n"
        f"Responsible body: {docs['responsible_body']}\n"
        f"Next step: {docs['next_steps_hint']}\n"
        f"Official source: {docs['source_url']}"
    )

In [56]:
# Tool-using agent
# This method uses the lookup tool first and then applies verification or hedging.

def tool_using_agent(service_name: str) -> str:
    """
    Build a final answer using the tool-based workflow.

    Parameters:
        service_name (str): The target service name.

    Returns:
        str: A grounded final answer.
    """

    result = lookup_service_tool(service_name)

    # Use hedging for mixed or unclear cases
    if result["found"] and result["jurisdiction_level"] in ["Mixed", "Unclear"]:
        return build_hedged_answer(service_name)

    # Use verification when the service is found
    if result["found"]:
        verification = verify_tool_result(result)
        return build_verified_answer(result, verification)

    # Fallback if the tool does not find a match
    return (
        f"I could not find a clear match for '{service_name}'.\n\n"
        "Please check the official municipal website or provide more details."
    )

In [57]:
# Run multi-method comparison on the benchmark dataset

comparison_rows = []

for _, row in hard_benchmark_questions.iterrows():
    question = row["question"]
    expected_service = row["expected_service"]
    difficulty = row["difficulty"]
    case_type = row["type"]

    baseline_answer = keyword_baseline(question)
    retrieval_answer = retrieval_based_answer(expected_service)
    tool_answer = tool_using_agent(expected_service)

    comparison_rows.append({
        "question": question,
        "expected_service": expected_service,
        "difficulty": difficulty,
        "type": case_type,
        "keyword_baseline": baseline_answer,
        "retrieval_workflow": retrieval_answer,
        "tool_using_agent": tool_answer
    })

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

,question,expected_service,difficulty,type,keyword_baseline,retrieval_workflow,tool_using_agent
0,Who handles garbage pickup for apartments?,garbage pickup,medium,specific scenario,Garbage pickup is likely handled by the Region...,Service: garbage pickup\nDescription: Resident...,Service: garbage pickup\nJurisdiction level: R...
1,My road is icy. Who is responsible for snow cl...,Snow removal,hard,mixed ownership,Snow or road-related issues may depend on the ...,Service: Snow removal\nDescription: Snow clear...,Service: Snow removal\nJurisdiction level: Cit...
2,How do I renew my passport quickly?,Passport renewal,medium,federal service,Passport services are handled by the Governmen...,Service: Passport renewal\nDescription: Passpo...,Service: Passport renewal\nJurisdiction level:...
3,Where can I update my OHIP card after moving?,OHIP card,hard,provincial service,OHIP or health card services are handled by th...,Service: OHIP card\nDescription: Ontario healt...,Service: OHIP card\nJurisdiction level: Provin...
4,I need housing help urgently. Who should I con...,Housing support,hard,unclear case,Housing support may involve different agencies...,Service: Housing support\nDescription: Housing...,The request for 'Housing support' may involve ...


In [58]:
# Compact benchmark view

comparison_df[["question", "expected_service", "difficulty", "type"]]

,question,expected_service,difficulty,type
0,Who handles garbage pickup for apartments?,garbage pickup,medium,specific scenario
1,My road is icy. Who is responsible for snow cl...,Snow removal,hard,mixed ownership
2,How do I renew my passport quickly?,Passport renewal,medium,federal service
3,Where can I update my OHIP card after moving?,OHIP card,hard,provincial service
4,I need housing help urgently. Who should I con...,Housing support,hard,unclear case


In [59]:
# Print full comparison results in a readable format

for i, row in comparison_df.iterrows():
    print("=" * 100)
    print(f"Example {i+1}")
    print(f"Question: {row['question']}")
    print(f"Expected Service: {row['expected_service']}")
    print(f"Difficulty: {row['difficulty']}")
    print(f"Type: {row['type']}\n")

    print("Keyword baseline:")
    print(row["keyword_baseline"])
    print("\nRetrieval-based workflow:")
    print(row["retrieval_workflow"])
    print("\nTool-using agent:")
    print(row["tool_using_agent"])
    print()

Example 1
Question: Who handles garbage pickup for apartments?
Expected Service: garbage pickup
Difficulty: medium
Type: specific scenario

Keyword baseline:
Garbage pickup is likely handled by the Region of Waterloo.

Retrieval-based workflow:
Service: garbage pickup
Description: Resident garbage collection services.
Responsible body: Region of Waterloo Waste Management
Next step: Please check the official source for the next step.
Official source: https://www.regionofwaterloo.ca/

Tool-using agent:
Service: garbage pickup
Jurisdiction level: Region
Responsible body: Region of Waterloo Waste Management
Description: Resident garbage collection services.
Source URL: https://www.regionofwaterloo.ca/

Confidence: high
Next step: Use the official source URL to confirm details and contact the responsible body.

Example 2
Question: My road is icy. Who is responsible for snow clearing?
Expected Service: Snow removal
Difficulty: hard
Type: mixed ownership

Keyword baseline:
Snow or road-relate

In [60]:
# Qualitative comparison summary

summary_df = pd.DataFrame([
    {
        "method": "Keyword baseline",
        "strength": "Fast and simple",
        "limitation": "Less grounded and less structured"
    },
    {
        "method": "Retrieval-based workflow",
        "strength": "Uses fuller retrieved service documentation",
        "limitation": "Still depends on catalog coverage"
    },
    {
        "method": "Tool-using agent",
        "strength": "Grounded, structured, and supports hedging",
        "limitation": "Requires well-designed tools and catalog entries"
    }
])

summary_df

,method,strength,limitation
0,Keyword baseline,Fast and simple,Less grounded and less structured
1,Retrieval-based workflow,Uses fuller retrieved service documentation,Still depends on catalog coverage
2,Tool-using agent,"Grounded, structured, and supports hedging",Requires well-designed tools and catalog entries


In [62]:
# Optional simple metrics
# Here we estimate whether the expected service name appears in each output.

def contains_expected_service(answer: str, expected_service: str) -> bool:
    return expected_service.lower() in answer.lower()

metrics_rows = []

for _, row in comparison_df.iterrows():
    expected_service = row["expected_service"]

    metrics_rows.append({
        "expected_service": expected_service,
        "keyword_hit": contains_expected_service(row["keyword_baseline"], expected_service),
        "retrieval_hit": contains_expected_service(row["retrieval_workflow"], expected_service),
        "tool_hit": contains_expected_service(row["tool_using_agent"], expected_service)
    })

metrics_df = pd.DataFrame(metrics_rows)
metrics_df

,expected_service,keyword_hit,retrieval_hit,tool_hit
0,garbage pickup,True,True,True
1,Snow removal,False,True,True
2,Passport renewal,False,True,True
3,OHIP card,False,True,True
4,Housing support,True,True,True


In [63]:
# Aggregate simple hit-rate metrics

aggregate_metrics = pd.DataFrame([
    {
        "method": "Keyword baseline",
        "hit_rate": metrics_df["keyword_hit"].mean()
    },
    {
        "method": "Retrieval-based workflow",
        "hit_rate": metrics_df["retrieval_hit"].mean()
    },
    {
        "method": "Tool-using agent",
        "hit_rate": metrics_df["tool_hit"].mean()
    }
])

aggregate_metrics

,method,hit_rate
0,Keyword baseline,0.4
1,Retrieval-based workflow,1.0
2,Tool-using agent,1.0


### Evaluation Summary

This benchmark comparison shows that system design matters.

A simple keyword baseline can provide rough answers, but retrieval and tool-based workflows improve grounding, structure, and reliability.

The tool-using agent performs best in this notebook because it supports:

- structured service lookup  
- verification of grounded answers  
- cautious handling of mixed or unclear cases

### Comparison Result

The keyword baseline is fast, but it is less grounded because it relies on general keyword rules instead of structured service data.

The retrieval-based workflow is stronger because it returns fuller service documentation and next-step guidance.

The tool-using agent is the strongest workflow in this notebook because it combines structured lookup, verification, and hedging for unclear or mixed cases.

##  Evaluation Layer: Metrics and Experiment Tracking

In this section, we summarize benchmark performance and track simple experiment metrics.

We record:

- per-example predictions  
- hit-rate metrics  
- method comparison results  

This helps make the workflow measurable, reproducible, and easier to improve over time.

In [64]:
# Save experiment results

comparison_df.to_csv("artifacts/comparison_results.csv", index=False)
aggregate_metrics.to_csv("artifacts/metrics_summary.csv", index=False)

print("Experiment files saved in artifacts/")

Experiment files saved in artifacts/


##  Evaluation Layer: Pipeline and DVC Integration

If this workflow is expanded into a larger ML system, retrieval and indexing steps should be tracked in a reproducible pipeline.

DVC can be used to version:

- benchmark datasets  
- cleaned catalogs  
- retrieval indexes  
- experiment outputs  
- evaluation metrics

This helps keep experiments organized and makes future improvements easier to compare.

In [65]:
# Example DVC pipeline stages (conceptual)

dvc_stages = {
    "prepare_catalog": "python prepare_catalog.py",
    "build_index": "python build_retrieval_index.py",
    "run_evaluation": "python evaluate_pipeline.py",
    "save_metrics": "python export_metrics.py"
}

pd.DataFrame(
    [{"stage": k, "command": v} for k, v in dvc_stages.items()]
)

,stage,command
0,prepare_catalog,python prepare_catalog.py
1,build_index,python build_retrieval_index.py
2,run_evaluation,python evaluate_pipeline.py
3,save_metrics,python export_metrics.py


## Talking Points

### 1. What is the smallest useful definition of an AI agent?

In our opinion, an AI agent is more than a chatbot that only gives text answers.  
A useful agent should know what action to take, such as retrieving information, calling a tool, and then generating the final response.

### 2. When should you prefer retrieval over memory?

We should prefer retrieval when the answer depends on real facts, official records, or information that can change over time.  
In this project, using the local service catalog is safer than relying only on model memory.

### 3. When is a function call better than asking the model directly?

A function call is better when we already have a trusted source or a fixed process.  
For example, checking service ownership or giving next steps is more reliable through tools connected to the dataset.

### 4. What does a grounded answer mean in a civic context?

A grounded answer means the response is supported by actual service data and official source links, not guessing.  
This is important because wrong civic information may send users to the wrong department.

### 5. Why might a single-agent system outperform a multi-agent system on a student project?

For a student project, a single-agent system is usually easier for us to build, test, and debug.  
A multi-agent system may sound advanced, but it can add extra complexity without much benefit.

### 6. What parts of this architecture should become DVC pipeline stages later?

The data cleaning step, retrieval or indexing step, evaluation step, and report generation step should become DVC stages later.  
These parts create outputs and metrics, so version tracking is useful.

### 7. Which parts of this problem are classification problems?

Predicting jurisdiction level is a classification task.  
Deciding the responsible body is also similar to classification.  
Another example is labeling requests as clear, mixed, or unclear.

### 8. Which parts require retrieval?

Retrieval is needed when the system matches the user question to the correct service record.  
It is also needed when finding descriptions and official source links.

### 9. Which parts benefit from tool use?

Tool use helps tasks that should be accurate and repeatable, such as searching the catalog, checking ownership, and preparing next-step guidance.

### 10. Which outputs should be deterministic and machine-readable?

Outputs such as service name, jurisdiction level, responsible body, confidence score, source links, and next steps should be machine-readable.  
This makes the workflow easier for us to evaluate and improve.

## Conclusion

Overall, this notebook shows how a local civic-service dataset can be transformed into a practical AI agent workflow.

By combining retrieval, tool functions, verification, and evaluation, we created a system that is more grounded, structured, and measurable than a prompt-only approach.

Although the project is lightweight, it demonstrates important real-world ideas such as reliable routing, benchmark testing, experiment tracking, and reproducible workflow design.

This project also helped us understand that strong AI systems depend not only on models, but also on good data and good system design.